# WHY GENERATORS?

Consider the scenario where we need to process 1 million numbers. Then, the naive approach is:

```python
def get_nums(n):
    result = []
    for i in range(n):
        result.append(i)
    return result

for num in get_nums(1_000_000):
    process(num)
```

This builds the entire list in the memory before we can even process the 1st number. For large data - log files, database rows, infinite sequences - this is wasteful and outright impossible.

Generators, on the other hand, return one values one at a time, on demand, thus allowing us to process data without storing the entire data. 

In other words, instead of computing the entire sequence of data before processing, generators allow us to delay computation and only perform the computation of 1 element at a time, whenever required, using the `next()` method.


---

# INTRODUCTION TO GENERATORS

A generator is a special and easier way to write iterators in python, by using `yield` keyword. Instead of manually defining a class and creating:
- static variables,
- `__iter__` and `__next__` methods
- `StopIteration` logic
python handles everything automatically when using generators.

### Iterator using Class

In [4]:
class CountUp:
    def __init__(self, stop):
        self.current = 0
        self.stop = stop
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration()
        
        value = self.current
        self.current += 1
        return value

c = CountUp(3)
print(c)
for num in c:
    print(num)

0
1
2


### Iterator using Generator

In [6]:
def count_up(n):
    for i in range(n):
        yield i

c = count_up(3)
print(c)
for num in c:
    print(num)

<generator object count_up at 0x000002666DB7D8A0>
0
1
2


# WORKING OF GENERATORS

## How a Normal Function Works

In normal funcitons, code is executed once from top to bottom, whenever the function is called. Consider the following function:

In [ ]:
def f():
    print("A")
    print("B")
    return 10

a = f()
print(f"returned value: {a}")

A
B
returned value: 10


When the function is called, python does the following:
- create a stack frame
- execute everything immediately
- return output (if anything is to be returned)
- destroy the frame after `return`.

The function is dead afterwords.

## How Generators Work

Generators are functions that can pause their own execution mid-way and resume where they left off whenever required. The only syntatic difference in using generators (instead of regular functions) is the keyword `yield` instead of (or alongside) `return`.

``` python
def count_up(n):
    for i in range(n):
        yield i
```

when python sees `yield` inside a funciton, it transforms the entire funciton into a **generator funciton**. Calling such a function does not run any code, but rather, returns a **generator object**. 

In [10]:
def g():
    print("A")
    yield 1
    print("B")
    yield 2

In [11]:
gen = g()
print(gen)

<generator object g at 0x000002666D9E6F80>


This object contains:
- the function code
- instruction pointer
- local variables
- evaluation stack
- execution context

i.e.,
``` bash
Generator Object
├── code object
├── current position
├── local variables
├── stack state
└── suspended execution frame
```

Since it just creates the object, nothing has been executed yet.

In order to run the function body, we use the `next()` method. This function starts the execution of the function untill the 1st `yield`:

In [12]:
print(next(gen))
print(next(gen))

A
1
B
2


At this moment:
- value `1` is emitted outside (returned)
- execution freezes
- stack frame is preserved
- local variables remain alive
- instruction pointer stays at yield location

i.e., the function is *suspended*, not *terminated*.

The `next(gen)` is a method defined for generator objects, which is used to resume the code execution from the previous `yield`, untill the next `yield`.

Generators raise a `StopIteration` exception once the sequence is complete.

**NOTE:** The generators local state is preserved between calls. Local variables, loop counter, etc. - the entire local state of the function is preserved. Whenever we call `next()`, the execution begins exactly from after the `yield` statement

---

# MISCELLENOUS CONCEPTS

## Using Generators with `for` loops

Generators are rarely used with `next()`. Python's `for` loop does it automatically - it calls `next()` repeatedly untill it catches `StopIteration`, then stops cleanly.

In [3]:
for num in count_up(3):
    print(num)

0
1
2


This works because the generator object implements python's iterator protocol, i.e., it defines `__iter__()` and `__next__()` methods.

**NOTE:** Any object that follows pythons iterator protocol - i.e., defines `__iter__()` and `__next__()` methods, can be used in a `for` loop.

---

## Generator Expressions - Inline Generators

Just like comprehensions create lists, generator expressions create generators. The syntax of generator expressions is the same as list comprehensions, but using `()` instead of `[]`.

```python
squares_list = [x*x for x in range(1000)]   # builds 1000 items in memory now
squares_gen = (x*x for x in range(1000))    # builds nothing
```

The generator expression computes squares only when required. Using generator expressions is the fastest way to write simple generator for use inside functions like `sum()`, `max()`, etc.

```python
total = sum(x**2 for x in range(1_000_000))     # need not store all values.
```

Generator expression can be converted to list easily using `list()`:

In [4]:
squares_gen = (x*x for x in range(10))
squares_list = list(squares_gen)
print(squares_list)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


---

## `yield from`: Delegating to another Generator

`yield from` is used when we want one generator to yield values from another generator (or any iterable) in a clean way, instead of looping manually. For e.g., instead of:

```python
    for item in a:
        yield item

    for item in b:
        yield item

list(chain([1, 2], [3, 4]))   # [1, 2, 3, 4]
```

we simply use `yield from` as follows:

In [13]:
def chain(a, b):
    yield from a    # yields one value from first list, i.e., list a
    yield from b    # after all yields are done from a, chain starts yielding from list b

print(list(chain([1, 2], [3, 4])))

[1, 2, 3, 4]


---

## Two way communication: `send()`

`yield` is not just an output mechanism. It's also used to inject values into the generators. To inject values into the generator, we use:
```python
gen.send(val)
```

Also, whenever we advance the generator using `next(gen)`, python automatically passes None as send - i.e., `next(gen)` is internally evaluated to:
```python
gen.send(None)
```

We can use the `send()` function to inject values into the generator as follows:

In [15]:
def accumulator():
    total = 0
    while True:
        value = yield total    # pause, hand out total, receive next value
        total += value

In [17]:
gen = accumulator()
print(next(gen))       # run to first yield. evaluate LHS and yield out 10. Execution stopped mid-way of statement, just before assignment
print(gen.send(10))    # sends 10 into the generator, inplace of yield. Now resume execution and complete assignment.
print(gen.send(5))     # sends 5 → total becomes 15, yields 15

0
10
15


Note the 2nd line:
```python
print(next(gen))
```
is needed to reach the execution of the first yield, where we actually want to inject the values. This step is called **primming the generator**.

> *Since generators behave like two-way communication channel they're often called **coroutines** in advanced contexts.*

## Closing a generator: `close()` and `throw()`

Generators are cleanly closed by using these 2 *control* methods:

1. `gen.close()`: tells the generator that it's done. Python throws a `GeneratorExit` exception inside it, which triggers any finally cleanup blocks

2. `gen.throw(exc)`: injects an exception at the *point where the generator is paused*, as if the `yield` itself raised it. Useful for error signalling in *coroutine style pipelines*

## Generator Pipelines

One of the most elegant uses of generators is chaining them into data pipelines - each stage consumes from the previous one, and nothing runs until the final consumer pulls a value:

In [19]:
def read_lines(filename):
    with open(filename) as f:
        yield from f

def filter_errors(lines):
    for line in lines:
        if "ERROR" in line:
            yield line

def parse(lines):
    for line in lines:
        yield line.strip().split(" - ")

# pipeline = parse(filter_errors(read_lines("app.log")))

# for entry in pipeline:
#     print(entry)

This is memory-efficient even on a 10GB log file, as only one line exists in the memory at a time.